In [6]:
puts `pwd`
puts `ls -l raw-data`

/home/osboxes/CODE/SIMPATHIC2/SKG_Mapping/demokritos
total 9556
-rw-rw-r-- 1 osboxes osboxes   34777 Apr 15 13:32 Demokritos-KG-information.xlsx
-rw-rw-r-- 1 osboxes osboxes  402701 Apr 15 13:32 Disease-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes  125055 Apr 15 13:32 Disease-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes   20901 Apr 15 13:32 Disease-Gene triples.tsv
-rw-rw-r-- 1 osboxes osboxes       0 Apr 15 16:21 disease_list.txt
-rw-rw-r-- 1 osboxes osboxes  207331 Apr 15 13:32 Disease-Therapeutic_Area.tsv
-rw-rw-r-- 1 osboxes osboxes   82213 Apr 15 13:32 Drug-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes 7769112 Apr 15 13:32 Drug-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  111643 Apr 15 13:32 Drug-Drug_type.tsv
-rw-rw-r-- 1 osboxes osboxes   87477 Apr 15 13:32 Drug-Gene triples.tsv
drwxrwxr-x 2 osboxes osboxes    4096 Apr  7 15:55 Feb 2026
-rw-rw-r-- 1 osboxes osboxes  684226 Apr 15 13:32 Gene-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes   57380 Apr 15 13:32 Gene-Drug

$ head -2 raw-data/Disease-Disease\ triples.tsv 
Disease1	Disease1_id	RELATION	PROVENANCE	Disease2	Disease2_id
Empty Sella Syndrome	C0014008	IS_A		Other and unspecified anterior pituitary hyperfunction	C0029493
COLUMN 2   AND  COLUMN 6

$ head -2 raw-data/Disease-Drug\ triples.tsv 
Disease	Disease_id	RELATION	PROVENANCE	Drug	Drug_id
Glutaric aciduria, type 1	C0268595	PRODUCES	["20032085_fullText_1"]	Lysine	C0024337
COLUMN 2

$ head -2 raw-data/Disease-Gene\ triples.tsv
Disease	Disease_id	RELATION	PROVENANCE	Gene	Gene_id
Hereditary Diseases	C0019247	ASSOCIATED_WITH	["34120625_fullText_99"]	NGLY1 gene	C1425023
COLUMN 2

$ head -2 raw-data/Disease-Therapeutic_Area.tsv 
Disease	Disease_id	Therapeutic_area	Therapeutic_area_id	
Third cranial nerve disorder	C0271353	Cranial nerve diseases	C0010266
COLUMN 2

$ head -2 raw-data/Gene-Disease\ triples.tsv 
Gene	Gene_id	RELATION	PROVENANCE	Disease	Disease_id
ZIC1 gene	C1421581	ASSOCIATED_WITH		Craniosynostosis	C0010278
COLUMN 6

$ head -2 raw-data/Drug-Disease\ triples.tsv 
Drug	Drug_id	RELATION	PROVENANCE	Disease	Disease_id
Lysine	C0024337	CAUSES	["23658800_fullText_123"]	Hypoglycemia	C0020615
COLUMN 6

In [48]:
puts `awk -F'\t' '{print $2}' "raw-data/Disease-Disease triples.tsv" | sort | uniq > disease_list.txt`
puts `cat disease_list.txt | wc -l`


3317


In [49]:
# Note it is now append, not write!
puts `awk -F'\t' '{print $6}' "raw-data/Disease-Disease triples.tsv" | sort | uniq >> disease_list.txt`
puts `cat disease_list.txt | wc -l`


4489


In [50]:
# Note it is now append, not write!
puts `awk -F'\t' '{print $2}' "raw-data/Disease-Drug triples.tsv" | sort | uniq >> disease_list.txt`
puts `cat disease_list.txt | wc -l`


4855


In [51]:
# Note it is now append, not write!
puts `awk -F'\t' '{print $2}' "raw-data/Disease-Gene triples.tsv" | sort | uniq >> disease_list.txt`
puts `cat disease_list.txt | wc -l`


4974


In [46]:
# Note it is now append, not write!
# This file is badly formatted - skip it
# puts `awk -F'\t' '{print $2}' "raw-data/Disease-Therapeutic_Area.tsv" | sort | uniq >> disease_list.txt`
# puts `cat disease_list.txt | wc -l`


8014


In [52]:
# Note it is now append, not write!
puts `awk -F'\t' '{print $6}' "raw-data/Gene-Disease triples.tsv" | sort | uniq >> disease_list.txt`
puts `cat disease_list.txt | wc -l`


6987


In [53]:
# Note it is now append, not write!
puts `awk -F'\t' '{print $6}' "raw-data/Drug-Disease triples.tsv" | sort | uniq >> disease_list.txt`
puts `cat disease_list.txt | wc -l`


7307


In [54]:
# add header first
puts `(echo "demokritos_umls"; cat disease_list.txt | sort | uniq) > disease_list_uniq.txt`
puts `head -5 disease_list_uniq.txt `



demokritos_umls
C0000744
C0000774
C0000833
C0000889


In [55]:
#!/usr/bin/env ruby

require 'csv'
require 'rest-client'
require 'json'
require 'uri'

# Configuration
INPUT_FILE = './disease_list_uniq.txt'
OUTPUT_FILE = "./maps/2026-demokritos-disease-mondo.map"
NO_MATCH_FILE = './maps/2026-unmapped-cuis.csv'
BATCH_SIZE = 50  # Adjust if needed (e.g., test with smaller batches if server limits)

unless File.exist?(INPUT_FILE)
  abort "Error: #{INPUT_FILE} not found!"
end

# Read CSV and collect CUIs (no uniq here, as per your code; add .uniq if duplicates are an issue)
rows = CSV.read(INPUT_FILE, headers: true)
cuis = rows.map { |row| row['demokritos_umls'].strip }

puts "Total CUIs: #{cuis.size}"

if cuis.empty?
  abort "Error: No CUIs found in #{INPUT_FILE}"
end

# Map CUI → MONDO ID
mondo_map = {}

cuis.uniq.each_slice(BATCH_SIZE) do |batch|  # Use uniq to avoid duplicate queries; slice for batching
  puts "Querying Monarch API for batch of #{batch.size} CUIs..."

  entity_ids = ""
  batch.each do |cui|
    entity_ids += "entity_id=UMLS:#{cui}&"
  end

  response = RestClient.get(
    "https://api-v3.monarchinitiative.org/v3/api/mappings?#{entity_ids}format=json&limit=500&offset=0"
  )

#   warn "Request URL: #{response.request.url}"

  data = JSON.parse(response.body)

  data['items'].each do |item|
    if item['object_id'] =~ /^UMLS:(C\d+)/
      cui = $1
      mondo_id = item['subject_id']  # e.g. "MONDO:0018940"
      mondo_term = item['subject_label']  # e.g. "Leigh Syndrome"
      mondo_map[cui] = {id: mondo_id, term: mondo_term}
    end
  end
#     break # do just one iteration
end

# Process rows
output_rows = []
no_match_rows = []

rows.each do |row|
  cui = row['demokritos_umls'].strip
  if (mondo_id = mondo_map.dig(cui, :id))
#       warn mondo_id, mondo_map[cui][:term], mondo_map[cui].inspect
      prefname = mondo_map.dig(cui, :term)
    # BioPortal PURL format
    row['prefname'] = prefname
    row['mondo'] = "http://purl.obolibrary.org/obo/#{mondo_id.gsub(':', '_')}"
    output_rows << row
  else
    no_match_rows << row
  end
end

# Write output CSV (with mondo  and prefname column)
headers = rows.headers
warn headers.inspect
CSV.open(OUTPUT_FILE, 'w', write_headers: true, headers: headers) do |csv|
  output_rows.each { |row| csv << row }
end
puts "Wrote #{output_rows.size} rows with MONDO mappings to #{OUTPUT_FILE}"

# Write unmatched rows
if no_match_rows.any?
  CSV.open(NO_MATCH_FILE, 'w', write_headers: true, headers: rows.headers) do |csv|
    no_match_rows.each { |row| csv << row }
  end
  puts "Wrote #{no_match_rows.size} unmatched rows to #{NO_MATCH_FILE}"
else
  puts "All CUIs found a MONDO match!"
end

Total CUIs: 4415
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 CUIs...
Querying Monarch API for batch of 50 C

["demokritos_umls", "prefname", "mondo"]


Wrote 3440 rows with MONDO mappings to ./maps/2026-demokritos-disease-mondo.map
Wrote 975 unmatched rows to ./maps/2026-unmapped-cuis.csv


In [58]:
puts `head -2 ./maps/2026-demokritos-disease-mondo.map`

# Note that C0001144 is "Acne" - not a disease per se... maybe a phenotype?
# Note that Acne appears in the HPO, BUT ALSO IN MONDO:  http://purl.obolibrary.org/obo/MONDO_0011438
# It seems that the CUI mapping to Mondo is incomplete :-(
# May need to do a backup keyword search :-P
puts `head -2 ./maps/2026-unmapped-cuis.csv`


demokritos_umls,prefname,mondo
C0000744,abetalipoproteinemia,http://purl.obolibrary.org/obo/MONDO_0008692
demokritos_umls,prefname,mondo
C0001144


In [ ]:
#